# 03. Continuación de Entrenamiento, Compatibilidad, Transferencia y Encadenamiento

Demuestra los cuatro mecanismos de reutilización implementados en `pinn_thermal.core.transfer`:
1. **Reanudación exacta** (`resume_from`).
2. **Transferencia de pesos entre modelos compatibles** (`WeightTransfer` incluso entre archivos JSON ligeramente distintos).
3. **Transferencia funcional universal** (`FunctionalTransfer` entre arquitecturas incompatibles, p. ej. `M0 -> M1` o `RBF-FD -> CUSPR`).
4. **Encadenamiento directo** (`ChainedPINN`: un modelo previo como campo base residual/feature).

In [1]:
import sys
from pathlib import Path

# Localiza PINN_V4 automáticamente aunque se ejecute desde dentro de notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Raíz PINN_V4:", ROOT)

Raíz PINN_V4: C:\usr\ths_mia_fiis\tesis_MIA_lk\tesis_MIA_2026-2_pinn


In [2]:
import pinn_thermal.models
from pinn_thermal.core.base_model import ModelRegistry
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.core.transfer import CompatibilityChecker
from pinn_thermal.visualization import CaseEditorHelper
from frontend.state import WorkspaceStore

store = WorkspaceStore(ROOT / "frontend" / "workspace", ROOT / "cases")
prob_a = store.load_problem("mms_constant")

# 1) Entrenar un modelo base M1
art_base = PINNTrainer(TrainConfig(adam_steps=120, lbfgs_steps=10, width=20, depth=3)).train(
    prob_a, "M1", store.runs_dir / "nb03_base_M1", overwrite=True
)

# 2) Modificar el JSON (nueva física/fuente) manteniendo características críticas geométricas compatibles
ed = CaseEditorHelper(prob_a.doc)
ed.set_metadata("mms_constant_perturbado", "MMS perturbado compatible", "mms")
ed.doc["regions"][0]["source"] = 1.15 * float(prob_a.doc["regions"][0]["source"]) if isinstance(prob_a.doc["regions"][0]["source"], (int, float)) else prob_a.doc["regions"][0]["source"]
prob_b = ed.build_problem()

cand_m1 = ModelRegistry.build(prob_b, "M1", width=20, depth=3)
rep_ok = CompatibilityChecker.check(art_base, cand_m1)
print("Compatibilidad M1 -> M1 (distinto JSON):", rep_ok.to_dict())

# Continuar con transferencia de pesos sobre prob_b
art_warm = PINNTrainer(
    TrainConfig(adam_steps=80, lbfgs_steps=10, width=20, depth=3, warm_start_from=art_base.source_dir, transfer_mode="weights")
).train(prob_b, "M1", store.runs_dir / "nb03_warm_M1", overwrite=True)
print("Reporte warm-start pesos:", art_warm.result["transfer_report"])

# 3) Transferencia funcional entre arquitecturas distintas: M1 -> CUSPR
art_xf = PINNTrainer(
    TrainConfig(adam_steps=80, lbfgs_steps=10, width=24, depth=3, warm_start_from=art_base.source_dir, transfer_mode="functional_pretrain", transfer_steps=60)
).train(prob_b, "CUSPR", store.runs_dir / "nb03_func_CUSPR", overwrite=True)
print("Reporte transferencia funcional:", art_xf.result["transfer_report"])

# 4) Encadenamiento directo residual: usa art_base como punto de partida T = T_prev + delta_T
art_chain = PINNTrainer(
    TrainConfig(adam_steps=80, lbfgs_steps=10, width=20, depth=3, chain_from=art_base.source_dir, chain_mode="residual")
).train(prob_b, "M1", store.runs_dir / "nb03_chained_M1", overwrite=True)
print("Modelo encadenado guardado:", art_chain.architecture, "->", art_chain.source_dir)

c:\Python312\Lib\site-packages\torch\optim\lbfgs.py:426: UserWarning: Aten Op fallback from XPU to CPU happends. This may have performance implications. If need debug the fallback ops please set environment variable `PYTORCH_DEBUG_XPU_FALLBACK=1`  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\xpu\ATen\RegisterXPU_0.cpp:54666.)
  gtd = flat_grad.dot(d)  # g * d
Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Compatibilidad M1 -> M1 (distinto JSON): {'compatible': True, 'level': 'weights_compatible', 'same_physics': False, 'matched_keys': ['region:domain -> region:domain'], 'skipped_keys': [], 'reasons': ['Características críticas y dimensiones de subredes compatibles.']}
Reporte warm-start pesos: {'method': 'weight_transfer', 'compatibility': {'compatible': True, 'level': 'weights_compatible', 'same_physics': False, 'matched_keys': ['region:domain -> region:domain'], 'skipped_keys': [], 'reasons': ['Características críticas y dimensiones de subredes compatibles.']}, 'pairs_copied': ['region:domain -> region:domain'], 'copied_tensors': 9}
Reporte transferencia funcional: {'transfer_method': 'pretrain', 'steps': 60, 'points': 256, 'initial_loss': 5.316286087036133, 'final_loss': 0.03921821713447571, 'rms_K': 5.773099422454834, 'max_K': 14.418119430541992, 'source_kind': 'pinn', 'source_architecture': 'M1'}
Modelo encadenado guardado: CHAINED(M1) -> C:\usr\ths_mia_fiis\tesis_MIA_lk\tesis_MIA_